## Ingest Fact Data into Bronze Layer

In [0]:
# Import Required Libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema,gold_schema)

In [0]:
# Widget
dbutils.widgets.text('catalog', 'shopvista', 'catalog')
dbutils.widgets.text('order_source', 'order_items', 'order_source')
dbutils.widgets.text('return_source', 'order_returns', 'return_source')
dbutils.widgets.text('shipment_source', 'order_shipments', 'shipment_source')

In [0]:
catalog = dbutils.widgets.get("catalog")
order_source = dbutils.widgets.get("order_source")
print(catalog, order_source)

## ORDER ITEMS

In [0]:
# show path to base, landing and processed data
base_path = f"s3://shopvista-sv/order_items/"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

# Define the table
bronze_table = f"{catalog}.{bronze_schema}.{order_source}"
silver_table = f"{catalog}.{silver_schema}.{order_source}"
gold_table =f"{catalog}.{gold_schema}.{order_source}"


In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_path")

In [0]:
# write to bronze table
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{order_source}")

In [0]:
# Moving files from source to processed directory
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )


## ORDER RETURNS

In [0]:
catalog = dbutils.widgets.get("catalog")
return_source = dbutils.widgets.get("return_source")
print(catalog, return_source)

In [0]:
# show path to base, landing and processed data
base_path = f"s3://shopvista-sv/order_returns/"
landing_path = f"{base_path}/landing/"
processed_path =f"{base_path}/processed/"

print("Base_path:", base_path)
print("landing_path", landing_path)
print("processed_path", processed_path)

# Define the table
bronze_table = f"{catalog}.{bronze_schema}.{return_source}"
silver_table = f"{catalog}.{silver_schema}.{return_source}"
gold_table =f"{catalog}.{gold_schema}.{return_source}"

In [0]:
# Read data
df = (spark.read
        .options(header=True, inferSchema=True)
        .csv(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_path")
)

In [0]:
# Write to Bronze layer
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{return_source}")

In [0]:

# Moving files from source to processed directory
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

## ORDER SHIPMENTS

In [0]:
catalog = dbutils.widgets.get("catalog")
shipment_source= dbutils.widgets.get("shipment_source") 
print(catalog, shipment_source)

In [0]:

# show path to base, landing and processed data
base_path = f"s3://shopvista-sv/order_shipments/" 
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print(f"base_path:", base_path)
print(f"landing_path", landing_path)
print(f"processed_path:", processed_path)


# Define table
bronze_table = f"{catalog}.{bronze_schema}.{shipment_source}"
silver_table = f"{catalog}.{silver_schema}.{shipment_source}"
gold_table = f"{catalog}.{silver_schema}.{shipment_source}"

In [0]:
# Read Data into df
df = (spark.read
      .options(header=True, infershema=True)
      .csv(f"{landing_path}/*csv")
      .withColumn("read_timestamp", F.current_timestamp())
      .select("*", "_metadata.file_name", "_metadata.file_path")
)

In [0]:
# Write to Bronze layer
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{shipment_source}")

In [0]:
# Moving files from source to processed directory
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/ {file_info.name}",
        True
    )